# Exploratory Data Analysis Before Cleaning

Inspect the full `merged_order_items` dataset from `data/interim` before deciding how to clean it.
Each row represents one order item, identified by `(order_id, order_item_id)`; order attributes can repeat across items.

- Use only the full merged dataset through the existing schema-aware loader. Do not reload raw tables or rerun merging.
- Keep all statuses, missing values and unusual records for inspection. Do not filter, fill, normalize or overwrite the input.
- Calculate diagnostic summaries in separate objects. Use the appropriate item, order or entity denominator for each question.
- Display tables and plots in this notebook. Do not export datasets or generate/update documentation.
- Business feature engineering, preprocessing and delivery-performance analysis belong to later work.


## Project setup

Run the notebook from the repository root or a directory inside the repository.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


# Reuse the project-root discovery convention from the merge notebook.
def find_project_root(start_path: Path) -> Path:
    """Find the project root containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.data_loader import load_data
from src.eda.checks import (
    build_entity_views,
    profile_entity_consistency,
    profile_keys,
    profile_order_composition,
    profile_text_formats,
    profile_timestamp_ordering,
    summarize_condition_by_group,
)
from src.eda.plots import (
    plot_frequency,
    plot_missingness_heatmap,
    plot_numeric_boxplot,
    plot_numeric_histogram,
)
from src.eda.profiling import (
    get_column_roles,
    limit_frequencies,
    profile_columns,
    profile_dataset,
    profile_frequencies,
    profile_numeric,
    profile_missing_patterns,
    profile_paired_missingness,
    profile_temporal_frequencies,
    profile_timestamp_coverage,
)
from src.paths import INTERIM_DIR
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA, ZIP_COLUMNS
from src.pipeline.contracts.validator import validate_source_keys

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data path: {INTERIM_DIR}")


## Load the full input

The default reader tries Parquet first, then falls back to CSV if needed, restoring the registered schema.
To select CSV directly, add `file_format="csv"` to the call below.

Keep `input_baseline` in memory for the final non-mutation check. Its row hashes include the index; columns and dtypes are recorded separately.
No baseline file or run metadata is written.


In [ ]:
df = load_data("merged_order_items", stage="interim")

# Record the loaded input without making a second copy of the full dataset.
input_baseline = {
    "shape": df.shape,
    "columns": df.columns.copy(),
    "dtypes": df.dtypes.copy(),
    "row_hashes": pd.util.hash_pandas_object(df, index=True).copy(),
}


## Input baseline

Reference counts describe the current Olist full merge, rather than a cleaning rule.
Show observed counts beside them; investigate differences without modifying the input to match.
The loader already validates the registered schema. A unique-order count does not change the item-level grain of `df`.


In [ ]:
baseline_summary = pd.DataFrame(
    [
        ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
        ["Unique orders", 98_666, df["order_id"].nunique()],
    ],
    columns=["Metric", "Reference", "Observed"],
)
baseline_summary["Matches reference"] = baseline_summary["Reference"].eq(
    baseline_summary["Observed"]
)
display(baseline_summary)

if not baseline_summary["Matches reference"].all():
    print("Reference counts differ. Check the input version before interpreting later profiles.")

print("Input baseline retained in memory for the final non-mutation check.")


## Initial inspection

Preview the loaded table before examining its structural profiles below.


In [ ]:
df.head()

## B. Structural profiles

All completeness counts on `df` use item rows. Repeated order, customer, product and seller attributes therefore receive item-level weighting here.
Entity views at the end of this section provide one row per observed entity for subsequent distributions.
`customer_id` identifies the customer record attached to a purchase; the full merge does not contain `customer_unique_id`, so customer counts are not unique-person counts.

### Dataset overview

Exact duplicate rows are different from repeated IDs. Participants include every row in a duplicate group; excess copies exclude its first row.


In [ ]:
dataset_profile = profile_dataset(df)
display(dataset_profile)


### Column roles and completeness

Roles reuse the registered merge schema. `source_table` is the first source contributing a column; `source_key` describes that source's grain, not a new key for the full table.
Identifiers and ZIP codes are not numeric measurements.

Missing means `isna()`. Blank means an empty or whitespace-only string and is counted separately.
Percentages use all input item rows. Distinct non-null counts include blanks and preserve original case/whitespace.


In [ ]:
column_roles = get_column_roles()
column_profile = profile_columns(df)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Registered column roles")
    display(column_roles)
    print("Completeness at item grain")
    display(column_profile)


### Key structure

Only `(order_id, order_item_id)` must be unique in the full item table. Repeated order/customer/product/seller IDs are expected at this grain.
Missing and blank key-row counts can overlap for composite keys; `invalid_rows` counts their union.
Distinct/repeated-key statistics use valid rows only. `excess_key_rows` is the number of valid rows beyond the first occurrence of each key.


In [ ]:
key_profile = profile_keys(df)
with pd.option_context("display.max_columns", None):
    display(key_profile)


### Entity attribute consistency

For each valid order/customer/product/seller ID, check every attribute contributed by its source table.
A missing value paired with a non-missing value is a conflict; an attribute missing on every row of the same ID is consistent but incomplete.
Null/blank ID rows are reported separately. Conflict percentages use valid distinct entities, rather than item rows.


In [ ]:
entity_consistency = profile_entity_consistency(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Consistency by entity")
    display(entity_consistency["summary"])
    print("Consistency by attribute")
    display(entity_consistency["attributes"])

if entity_consistency["conflicts"].empty:
    print("No conflicting entity attributes found.")
else:
    print("Conflicting entity attributes (all detected entity/attribute pairs)")
    display(entity_consistency["conflicts"])


### Validated entity views in memory

Build source-grain views only after checking complete keys and consistent entity attributes.
If a conflict or invalid key exists, view construction raises an error instead of silently selecting a value.

The views contain entities observed in the full merge, not all records from the raw source tables.
Removing repeated keys inside these temporary, validated views does not clean or modify `df`. Nothing is saved to disk.


In [ ]:
entity_views = build_entity_views(df)
entity_view_summary = pd.DataFrame(
    [
        [name, len(view), len(view.columns)]
        for name, view in entity_views.items()
    ],
    columns=["View", "Observed rows", "Columns"],
)
display(entity_view_summary)


## C. Distributions

### Numeric measurements

Use item rows for `price`/`freight_value` and one row per observed product for its physical measurements.
Identifiers and ZIP codes are excluded from numeric summaries. Units are BRL, grams and centimetres.

Statistics use finite, non-missing values. Missing and non-finite counts remain visible; percentages state their denominator.
Sample standard deviation uses `ddof=1`. IQR limits and fliers are diagnostic descriptions, not cleaning decisions.

Each measurement has a full-range histogram, a 1st–99th percentile display window and a full-range boxplot.
The window reports excluded display counts and does not change the table or statistics.


In [ ]:
numeric_units = {
    "price": "BRL", "freight_value": "BRL", "product_weight_g": "g",
    "product_length_cm": "cm", "product_height_cm": "cm", "product_width_cm": "cm",
}
numeric_profiles = {}

for view_name, view in entity_views.items():
    measurement_columns = column_roles.loc[
        column_roles["source_table"].eq(view_name)
        & column_roles["role"].eq("measurement"), "column",
    ].tolist()
    if not measurement_columns:
        continue
    summary = profile_numeric(view, measurement_columns)
    numeric_profiles[view_name] = summary
    indexed = summary.set_index("column")
    print(f"Numeric measurements | {view_name} | N={len(view):,}")
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print("Counts (missing_pct uses all rows)")
        display(indexed[[
            "rows", "finite", "missing", "missing_pct", "non_finite_non_null", "zero", "negative",
        ]].T)
        print("Distribution of finite values")
        display(indexed[[
            "min", "p01", "p05", "p25", "median", "p75", "p95", "p99", "max", "mean", "std",
        ]].T)
        print("IQR diagnostics (outside_iqr_pct uses finite values)")
        display(indexed[[
            "iqr", "iqr_lower", "iqr_upper", "below_iqr", "above_iqr", "outside_iqr_pct",
        ]].T)

    for column in measurement_columns:
        stats = indexed.loc[column]
        lower, upper = stats["p01"], stats["p99"]
        display_range = (float(lower), float(upper)) if pd.notna(lower) and pd.notna(upper) and lower < upper else None
        figure, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
        label = f"{column} ({numeric_units[column]})"
        plot_numeric_histogram(view[column], ax=axes[0], xlabel=label, title="Full range")
        plot_numeric_histogram(
            view[column], ax=axes[1], xlabel=label, value_range=display_range,
            title="1st–99th percentile window" if display_range else "Full range (no distinct quantile bounds)",
        )
        plot_numeric_boxplot(view[column], ax=axes[2], xlabel=label, title="Full range; 1.5-IQR whiskers")
        figure.suptitle(f"{column} | {view_name}")
        display(figure)
        plt.close(figure)


### Categories and ZIP codes

Use one row per order for status, per customer record for its location, per seller for its location and per product for category.
ZIP prefixes are categorical codes; high-frequency ZIP/city/category charts show the top 15 observed values.

Keep full frequency tables in `frequency_tables`. Display tables and charts use the same top values plus Other/Blank/Missing;
counts still sum to all rows in the named view. `pct_all` includes missing rows in its denominator.
Literal strings retain their original case and whitespace; special rows have a distinct `kind`.


In [ ]:
frequency_tables = {}
frequency_display_tables = {}

for view_name, view in entity_views.items():
    category_columns = column_roles.loc[
        column_roles["source_table"].eq(view_name)
        & column_roles["role"].isin(["category", "zip_code"]), "column",
    ].tolist()
    for column in category_columns:
        full_table = profile_frequencies(view[column])
        display_table = limit_frequencies(full_table, top_n=15)
        frequency_tables[(view_name, column)] = full_table
        frequency_display_tables[(view_name, column)] = display_table
        print(f"{column} | {view_name} | N={len(view):,}; top 15 plus special rows")
        with pd.option_context("display.max_rows", None):
            display(display_table)
        figure, axes = plot_frequency(
            display_table, top_n=None, title=f"{column} | {view_name}",
        )
        display(figure)
        plt.close(figure)


### Timestamp coverage and calendar distributions

Use order-level timestamps once per order. `shipping_limit_date` remains at item grain because one order can contain different item deadlines.
Inspect ranges, missingness, clock precision and calendar month/date/weekday/hour distributions for all six timestamps.

Calendar components use the stored clock. No timezone, business-day calendar, time-slot rule or delivery-performance feature is inferred.
Month/date tables contain observed calendar buckets; weekday/hour tables include all seven/24 buckets, including zero counts.
Missing timestamps remain a separate category and stay in each all-row denominator.

Store every calendar frequency table in `temporal_frequency_tables`. For readability, only the date table/chart uses the top 15 dates plus Other/Missing;
month, weekday and hour tables/charts preserve calendar order. Estimated dates at midnight should be interpreted as recorded date precision.


In [ ]:
timestamp_profiles = {}
temporal_frequency_tables = {}
temporal_display_tables = {}

for view_name, view in entity_views.items():
    timestamp_columns = column_roles.loc[
        column_roles["source_table"].eq(view_name)
        & column_roles["role"].eq("timestamp"), "column",
    ].tolist()
    if not timestamp_columns:
        continue
    coverage = profile_timestamp_coverage(view, timestamp_columns)
    timestamp_profiles[view_name] = coverage
    print(f"Timestamp coverage | {view_name} | N={len(view):,}")
    with pd.option_context("display.max_columns", None):
        display(coverage)

    for column in timestamp_columns:
        figure, axes = plt.subplots(2, 2, figsize=(16, 10), layout="constrained")
        for frequency, axis in zip(["month", "date", "weekday", "hour"], axes.flat):
            full_table = profile_temporal_frequencies(view[column], frequency)
            display_table = limit_frequencies(full_table, top_n=15) if frequency == "date" else full_table
            temporal_frequency_tables[(view_name, column, frequency)] = full_table
            temporal_display_tables[(view_name, column, frequency)] = display_table
            print(f"{column} | {view_name} | {frequency} | N={len(view):,}")
            with pd.option_context("display.max_rows", None):
                display(display_table)
            plot_frequency(
                display_table, top_n=None, horizontal=False, ax=axis,
                title="Top 15 calendar dates plus Other/Missing" if frequency == "date" else frequency.title(),
            )
        figure.suptitle(f"{column} | {view_name} | Stored clock")
        display(figure)
        plt.close(figure)


## D. Missingness in context

### Row patterns and native-grain completeness

Missing means `isna()`. Blank text remains a separate diagnostic in the existing completeness profile; it is not silently recoded as null.
Inspect the full 25-column table at item grain, then order/product/customer/seller views at their validated source grain.
These views cover only entities observed in the full merge.

Every row contributes exactly once to a missing-count distribution and one exact pattern. `()` means complete within that view's columns.
Percentages use all rows of the named view. Zero-count bins stay available in RAM; only observed count bins are displayed below.


In [ ]:
missingness_frames = {
    "full_items": df,
    **{name: view for name, view in entity_views.items() if name != "order_items"},
}
missingness_profiles = {}
native_column_profiles = {}

for view_name, view in missingness_frames.items():
    profile = profile_missing_patterns(view)
    missingness_profiles[view_name] = profile
    completeness = column_profile if view_name == "full_items" else profile_columns(view)
    native_column_profiles[view_name] = completeness
    print(f"{view_name} | N={len(view):,} | {len(view.columns)} selected columns")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        print("Columns with missing values (all counts at this view's grain)")
        incomplete = completeness.loc[completeness["missing"].gt(0), [
            "column", "rows", "missing", "missing_pct", "blank", "blank_pct",
        ]]
        if incomplete.empty:
            print("No missing values at this grain.")
        else:
            display(incomplete)
        print("Observed missing-count bins")
        display(profile["row_distribution"].loc[profile["row_distribution"]["rows"].gt(0)])
        print("All observed exact patterns; () means complete within the selection")
        display(profile["patterns"])


### Paired and directional missingness

Calculate every pair of the 25 full-table columns, retaining the full result in `missing_pair_profiles`.
Display pairs involving two columns that actually have missing observations, including pairs with zero co-missing rows.
Repeat at order and product grain so repeated items do not inflate entity-level counts.

Each pair partitions all rows into both missing, left-only missing, right-only missing and both present.
Heatmap row A/column B means **percentage of A-missing rows where B is also missing**. The denominator is A's missing count;
therefore the matrix can be asymmetric. An undefined denominator is NaN/grey, not 0%. Co-missing does not establish a cause.


In [ ]:
missing_pair_profiles = {}

for view_name in ["full_items", "orders", "products"]:
    view = missingness_frames[view_name]
    paired = profile_paired_missingness(view)
    missing_pair_profiles[view_name] = paired
    active = native_column_profiles[view_name].loc[
        native_column_profiles[view_name]["missing"].gt(0), "column",
    ].tolist()
    displayed_pairs = paired["pairs"].loc[
        paired["pairs"]["left"].isin(active) & paired["pairs"]["right"].isin(active)
    ]
    print(f"{view_name} | N={len(view):,} | {len(paired['pairs'])} total pairs; {len(displayed_pairs)} displayed")
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print("Pair counts (four states sum to N)")
        display(displayed_pairs[[
            "left", "right", "rows", "both_missing", "left_only_missing", "right_only_missing", "both_present",
        ]])
        print("Conditional percentages with explicit missing-row denominators")
        display(displayed_pairs[[
            "left", "right", "left_missing", "right_missing",
            "right_missing_given_left_pct", "left_missing_given_right_pct",
        ]])
    figure, axes = plot_missingness_heatmap(
        paired["conditional_pct"].loc[active, active],
        title=f"{view_name} | Also missing among rows missing the conditioning column",
    )
    display(figure)
    plt.close(figure)


### Missingness by observed groups

Compare timestamp missingness by order status. Also inspect any missing core timestamp by purchase month and customer state.
The four core timestamps follow the draft scope (purchase, approval, carrier handoff, customer delivery); no eligibility filter is applied.
Month is a temporary calendar grouping, not an engineered business time slot.

For product measurements, inspect catalogue completeness by category at product grain. For operational exposure, inspect missing product attributes
by seller/customer geography at item grain. Item-weighted groups can reflect repeated products and different product mixes.
Missing category/state/month/group keys stay in the denominators as observed missing-key groups.

Each output reports group rows, evaluated/unknown/affected counts and both evaluated-row/all-row percentages.
Here `isna()` conditions have no unknown values. The generic helper also preserves unknown conditions for later diagnostics.
Full group tables stay in `condition_profiles`. Seller/category displays show at most 15 groups ranked by affected count;
all-group N/affected totals and omitted-group counts are stated. A high percentage in a small group is not evidence of a cause.


In [ ]:
order_timestamp_columns = column_roles.loc[
    column_roles["source_table"].eq("orders") & column_roles["role"].eq("timestamp"), "column",
].tolist()
core_timestamp_columns = [
    column for column in order_timestamp_columns if column != "order_estimated_delivery_date"
]
product_measurement_columns = column_roles.loc[
    column_roles["source_table"].eq("products") & column_roles["role"].eq("measurement"), "column",
].tolist()

# Add only the observed customer state to a diagnostic order view from the full input.
# Check its key before grouping; never select an arbitrary customer state per order.
order_context = df.loc[:, [*entity_views["orders"].columns, "customer_state"]].drop_duplicates().copy()
validate_source_keys(order_context, "orders")
order_context = order_context.assign(
    purchase_month=order_context["order_purchase_timestamp"].dt.strftime("%Y-%m")
)
product_view = entity_views["products"]
core_missing = order_context[core_timestamp_columns].isna().any(axis=1)
product_measurements_missing = product_view[product_measurement_columns].isna().any(axis=1)
item_measurements_missing = df[product_measurement_columns].isna().any(axis=1)
item_category_missing = df["product_category_name"].isna()

condition_specs = [
    (column, "orders", order_context, order_context[column].isna(), ["order_status"], None)
    for column in order_timestamp_columns
]
condition_specs.extend([
    ("any_core_timestamp", "orders", order_context, core_missing, ["order_status"], None),
    ("any_core_timestamp", "orders", order_context, core_missing, ["purchase_month"], None),
    ("any_core_timestamp", "orders", order_context, core_missing, ["customer_state"], None),
    ("any_product_measurement", "products", product_view, product_measurements_missing, ["product_category_name"], 15),
    ("product_category_name", "items", df, item_category_missing, ["seller_id"], 15),
    ("product_category_name", "items", df, item_category_missing, ["customer_state"], None),
    ("product_category_name", "items", df, item_category_missing, ["seller_state"], None),
    ("any_product_measurement", "items", df, item_measurements_missing, ["seller_id"], 15),
    ("any_product_measurement", "items", df, item_measurements_missing, ["customer_state"], None),
])
condition_profiles = {}
condition_overview_rows = []

for target, grain, frame, condition, groups, display_limit in condition_specs:
    table = summarize_condition_by_group(frame, condition, groups)
    key = (target, grain, tuple(groups))
    condition_profiles[key] = table
    if display_limit is None:
        shown = table.sort_values(groups, na_position="last")
    else:
        shown = table.sort_values("affected_rows", ascending=False, kind="stable").head(display_limit)
    print(f"Missing {target} | grain={grain} | grouped by {groups}")
    print(
        f"All-group N={int(table['rows'].sum()):,}; affected={int(table['affected_rows'].sum()):,}; "
        f"showing {len(shown)} of {len(table)} groups; omitted groups={len(table) - len(shown)}"
    )
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(shown)
    condition_overview_rows.append([
        target, grain, tuple(groups), int(table["rows"].sum()),
        int(table["affected_rows"].sum()), len(table), len(shown),
    ])

condition_overview = pd.DataFrame(condition_overview_rows, columns=[
    "Missing target", "Grain", "Grouping", "Rows", "Affected rows", "Groups", "Displayed groups",
])
print("Conditional-profile overview")
with pd.option_context("display.max_rows", None):
    display(condition_overview)


### Order-status overview

Each cell is the percentage of orders in that status missing the indicated timestamp (or any core timestamp).
The denominator is the order count of that status, not the missing count used in the co-missing heatmaps above.
Statuses describe different lifecycle stages; this chart does not label incomplete records as errors or remove them.


In [ ]:
status_missing_rates = pd.concat(
    {
        target: condition_profiles[(target, "orders", ("order_status",))].set_index("order_status")["affected_pct_all"]
        for target in [*order_timestamp_columns, "any_core_timestamp"]
    },
    axis=1,
).sort_index(na_position="last")
status_missing_rates.index.name = "Order status (denominator: all orders in status)"
status_missing_rates.columns.name = "Missing timestamp / core set"
display(status_missing_rates)
figure, axes = plot_missingness_heatmap(
    status_missing_rates, title="Missing timestamp percentage within each order status",
)
display(figure)
plt.close(figure)


## E. Domain plausibility checks

### Original text, identifiers and location codes

Inspect strings without trimming, recasing, replacing characters or changing any input value.
Regexes describe structural conventions for this dataset: 32 lowercase hexadecimal characters for string IDs,
five ASCII digits for ZIP prefixes and two uppercase letters for states. They do not verify an actual address or an official state-code catalogue.
Numeric `order_item_id` is checked separately through order composition, not converted to text.

Whitespace, control/replacement characters, non-string values and regex mismatches are separate, potentially overlapping checks.
Regex percentages use original string rows, excluding missing/non-string values. Candidate variants share only a temporary stripped/casefolded
comparison key; they do not prove that two names refer to the same place or should be consolidated.

Profile all text columns at item grain to describe exposure, then at each native entity grain.
Full variant tables remain in RAM. At most 15 variant forms are displayed per view; example rows are capped at five per column/check.


In [ ]:
string_id_columns = column_roles.loc[
    column_roles["role"].eq("identifier") & column_roles["expected_dtype"].eq("string"), "column",
].tolist()
text_columns = column_roles.loc[column_roles["expected_dtype"].eq("string"), "column"].tolist()
text_shape_patterns = {
    **{column: r"[0-9a-f]{32}" for column in string_id_columns},
    **{column: r"[0-9]{5}" for column in ZIP_COLUMNS},
    "customer_state": r"[A-Z]{2}", "seller_state": r"[A-Z]{2}",
}
text_format_profiles = {}

for view_name, view in missingness_frames.items():
    selected = [column for column in text_columns if column in view.columns]
    profile = profile_text_formats(
        view, selected,
        patterns={column: text_shape_patterns[column] for column in selected if column in text_shape_patterns},
        sample_size=5,
    )
    text_format_profiles[view_name] = profile
    print(f"Original text formats | {view_name} | N={len(view):,}")
    indexed = profile["summary"].set_index("column")
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print("Value counts and text lengths (checks can overlap)")
        display(indexed[["rows", "missing", "string_rows", "non_string", "blank", "min_length", "max_length"]].T)
        print("Formatting diagnostics")
        display(indexed[[
            "edge_whitespace", "repeated_whitespace", "control_character",
            "replacement_character", "pattern_evaluated", "pattern_mismatch",
            "pattern_mismatch_pct", "candidate_variant_groups",
        ]].T)
        print("Regex declarations (no pattern means no regex evaluation)")
        display(profile["summary"][["column", "pattern"]])
    print(f"Candidate variant forms: showing {min(15, len(profile['variants']))} of {len(profile['variants'])}; full table stays in text_format_profiles")
    if not profile["variants"].empty:
        display(profile["variants"].head(15))
    if not profile["examples"].empty:
        print("Bounded examples; row_position refers to iloc within the named view")
        display(profile["examples"])


### Timestamp ordering at order grain

Inspect six physical timestamp comparisons and one estimated-date plausibility check.
A comparison is assessable only when both timestamps exist. Unknown comparisons remain NA, not False;
percentages use comparable orders and overlapping flags must not be added together.

Physical timestamps use their original clock precision. The estimated-before-purchase check uses calendar dates to avoid treating a same-day midnight
estimate as earlier merely because purchase occurred later in the day. Equality is not flagged.
No processing/transit durations, business-day calculations, lateness labels or time-slot features are created.


In [ ]:
timeline_profile = profile_timestamp_ordering(entity_views["orders"], sample_size=5)
with pd.option_context("display.max_columns", None):
    display(timeline_profile["summary"])
print(
    f"Orders with any ordering flag: {int(timeline_profile['conditions'].fillna(False).any(axis=1).sum()):,} "
    f"of {len(entity_views['orders']):,}; flags overlap"
)
if not timeline_profile["examples"].empty:
    print("Original timestamp examples (at most five per check)")
    with pd.option_context("display.max_columns", None):
        display(timeline_profile["examples"])

timeline_by_status = {}
for check in timeline_profile["conditions"].columns:
    table = summarize_condition_by_group(
        entity_views["orders"], timeline_profile["conditions"][check], ["order_status"],
    )
    timeline_by_status[check] = table
    if table["affected_rows"].sum():
        print(f"{check} by status; unknown_rows means one or both timestamps are absent")
        display(table)


### Measurement plausibility and item deadlines

Reuse the numeric profiles from section C. Review zero/negative physical measurements and non-finite values without changing them.
Zero freight is reported separately and is not automatically classified as invalid. IQR flags alone do not establish invalid measurements.

Inspect an item deadline earlier than purchase only when both raw timestamps exist. Keep this item-level diagnostic separate from seller delay:
no single deadline is selected for an order and no seller-delay label is created.


In [ ]:
measurement_review = pd.concat(
    [table.assign(grain=name) for name, table in numeric_profiles.items()], ignore_index=True,
)[["grain", "column", "rows", "finite", "missing", "non_finite_non_null", "zero", "negative"]]
display(measurement_review)

deadline_comparable = df["shipping_limit_date"].notna() & df["order_purchase_timestamp"].notna()
deadline_before_purchase = df["shipping_limit_date"].lt(
    df["order_purchase_timestamp"]
).astype("boolean").where(deadline_comparable, pd.NA)
deadline_by_status = summarize_condition_by_group(df, deadline_before_purchase, ["order_status"])
print("Item deadline before purchase by status (evaluated denominator: comparable item rows)")
display(deadline_by_status)
if deadline_before_purchase.any():
    print("First five flagged item rows; full diagnostic remains in deadline_before_purchase")
    display(df.loc[deadline_before_purchase.fillna(False), [
        "order_id", "order_item_id", "seller_id", "order_purchase_timestamp", "shipping_limit_date",
    ]].head(5))


### Order composition and deadline ambiguity

Inspect the number of items, products, sellers, deadlines and original location values within an order.
Multiplicity is descriptive: a multi-seller or multi-deadline order is not automatically an error.
Item numbering is consecutive only when unique positive integer item IDs cover 1..N without gaps.

Compare original state strings only where both are present and non-blank. These counts do not infer distance or create a route-type feature.
A mixture of equal/different state pairs means a later order-level route rule needs an explicit decision.
Min/max deadlines are descriptive ranges, not a chosen aggregation rule. A missing deadline is counted separately from distinct observed deadlines.
All order/group tables below stay in RAM; nothing is saved as a preprocessing dataset.


In [ ]:
order_composition = profile_order_composition(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(order_composition["summary"])
order_diagnostics = order_composition["orders"]
composition_frequency_tables = {}
for metric in ["item_rows", "products", "sellers", "deadlines"]:
    table = profile_frequencies(order_diagnostics[metric])
    composition_frequency_tables[metric] = table
    print(f"{metric} per order | N={len(order_diagnostics):,}")
    with pd.option_context("display.max_rows", None):
        display(table)

composition_example_masks = {
    "Multiple sellers": order_diagnostics["sellers"].gt(1),
    "Multiple deadlines": order_diagnostics["deadlines"].gt(1),
    "Mixed equal/different state pairs": order_diagnostics["items_with_equal_states"].gt(0) & order_diagnostics["items_with_different_states"].gt(0),
    "Incomplete item numbering": ~order_diagnostics["consecutive_item_numbers"],
}
for label, mask in composition_example_masks.items():
    print(f"{label}: {int(mask.sum()):,} orders; at most five examples")
    if mask.any():
        with pd.option_context("display.max_columns", None):
            display(order_diagnostics.loc[mask].head(5))

multi_deadline_orders = order_diagnostics.loc[order_diagnostics["deadlines"].gt(1), "order_id"].head(5)
print("Order-seller deadline ranges for the first five multi-deadline orders")
with pd.option_context("display.max_columns", None):
    display(order_composition["seller_deadlines"].loc[
        order_composition["seller_deadlines"]["order_id"].isin(multi_deadline_orders)
    ])


### Input preservation check

Compare the table against the baseline recorded immediately after loading, including column order, dtypes, values and row index.
The same baseline remains available for later EDA sections.


In [ ]:
assert df.shape == input_baseline["shape"], "Input shape changed during profiling."
pd.testing.assert_index_equal(df.columns, input_baseline["columns"])
pd.testing.assert_series_equal(df.dtypes, input_baseline["dtypes"])
pd.testing.assert_series_equal(
    pd.util.hash_pandas_object(df, index=True), input_baseline["row_hashes"],
)
print("Input preserved after domain plausibility checks.")